# 🤖 RecruitAI — Formación: Analizador de CVs con IA Generativa
 
**Objetivo:** Construir una app web completa que analiza CVs automáticamente usando la API de Google Gemini

---

## 📋 Índice

| Bloque | Contenido | Duración |
|--------|-----------|----------|
| **Bloque 1** | Contexto y conceptos: LLMs, APIs, prompts, JSON | 30-45 min |
| **Bloque 2** | Construcción guiada: pdf_loader → analyzer → app | 2h - 2h30 |
| **Bloque 3** | Demo en vivo + debate sobre sesgos y mejoras | 30 min |

---

# 🟦 BLOQUE 1 — Contexto y Conceptos
### ⏱️ 30-45 minutos

---

## 1.1 ¿Qué es un LLM y qué es una API?

Un **LLM (Large Language Model)** es un modelo de inteligencia artificial entrenado con enormes cantidades de texto. Es capaz de:
- Leer y comprender texto en lenguaje natural
- Generar texto coherente y contextual
- **Extraer información estructurada** de documentos no estructurados ← esto es lo que usaremos hoy

Ejemplos de LLMs populares:

| Empresa | Modelo |
|---------|--------|
| Google | Gemini 2.0 Flash ← **usaremos este** |
| Anthropic | Claude Sonnet |
| OpenAI | GPT-4o |
| Meta | LLaMA 3 |

Una **API (Application Programming Interface)** es una puerta de acceso a estos modelos desde nuestro código Python. En lugar de usar el chat en el navegador, enviamos texto por código y recibimos la respuesta también en código.

```
Tu código Python  →  [API de Gemini]  →  Respuesta del modelo
     (texto)                                   (texto / JSON)
```

---

## 1.2 El concepto de "Prompt"

Un **prompt** es la instrucción que le damos al modelo. Es el equivalente a lo que escribirías en el chat de Gemini o ChatGPT, pero desde código.

Hay una diferencia enorme entre un prompt vago y un prompt bien diseñado:

**❌ Prompt vago:**
```
"Analiza este CV"
```
→ El modelo responde lo que quiere, en el formato que quiere.

**✅ Prompt de extracción estructurada:**
```
"Actúa como auditor de RRHH. Extrae del siguiente CV:
 - nombre completo
 - años de experiencia
 - skills técnicas
 Devuelve ÚNICAMENTE un JSON válido, sin texto adicional."
```
→ El modelo devuelve exactamente lo que necesitamos, en el formato que necesitamos.

---

### 🛠️ Buenas prácticas de prompting

Un buen prompt combina varias técnicas. Aquí están las más importantes con ejemplos aplicados a nuestro proyecto:

---

#### 1. Asignar un rol (*Role Prompting*)

Decirle al modelo qué personaje debe adoptar mejora la calidad y el tono de la respuesta.

| | Ejemplo |
|---|---|
| ❌ Sin rol | `"Analiza este CV"` |
| ✅ Con rol | `"Actúa como un Auditor de Talento Senior con 10 años de experiencia en selección tech."` |

> El modelo "se pone en situación" y aplica criterios más alineados con lo que esperamos.

---

#### 2. Dar instrucciones específicas y sin ambigüedad

Cuanto más concreto, menos margen tiene el modelo para improvisar.

| | Ejemplo |
|---|---|
| ❌ Vago | `"Indica cuánta experiencia tiene"` |
| ✅ Específico | `"Calcula los años desde el primer trabajo real hasta hoy. NO cuentes becas, internships ni prácticas."` |

---

#### 3. Definir el formato de salida (*Output Format*)

Si necesitas procesar la respuesta con código, el formato debe ser explícito e inamovible.

| | Ejemplo |
|---|---|
| ❌ Sin formato | `"Extrae los datos del candidato"` |
| ✅ Con formato | `"Devuelve ÚNICAMENTE un JSON válido con esta estructura exacta, sin texto adicional, sin bloques de código markdown:"` |

```json
{
    "nombre_candidato": "string",
    "anos_experiencia_total": 0.0,
    "skills_tecnicas": ["skill1", "skill2"]
}
```

---

#### 4. Incluir un ejemplo del output esperado (*One-shot prompting*)

Mostrar al modelo un ejemplo concreto del resultado reduce drásticamente los errores de formato.

**❌ Sin ejemplo:**
```
"Devuelve el score como un número del 0 al 100"
```

**✅ Con ejemplo:**
```
"Devuelve el score como un número entero del 0 al 100.
Ejemplo: si el candidato encaja perfectamente → 95.
Si apenas cumple los requisitos mínimos → 30."
```

---

#### 5. Establecer reglas explícitas de negocio

Las reglas que no son obvias para el modelo deben escribirse explícitamente en el prompt.

**✅ Bloque de reglas:**
```
REGLAS IMPORTANTES:
- NO cuentes becas, internships o prácticas como experiencia real
- Si la ubicación no se menciona, escribe "No especificada"
- El score_match debe basarse SOLO en la oferta proporcionada, no en criterios generales
```

---

#### 6. Pedir razonamiento paso a paso (*Chain of Thought*)

Para tareas complejas como calcular un score, pedirle al modelo que razone antes de responder mejora la precisión.

**✅ Con Chain of Thought:**
```
"Para calcular el score_match sigue estos pasos:
1. Identifica los requisitos clave de la oferta
2. Comprueba cuántos cumple el candidato
3. Penaliza si faltan requisitos marcados como imprescindibles
4. Devuelve el score y explica en 'justificacion' los 2-3 factores decisivos"
```

---

#### Resumen: ¿cuándo usar cada técnica?

| Técnica | Cuándo usarla |
|---------|---------------|
| **Role Prompting** | Siempre. Define el "personaje" del modelo. |
| **Instrucciones específicas** | Siempre que haya riesgo de ambigüedad. |
| **Formato de salida** | Cuando la respuesta la procesa código (JSON, CSV...). |
| **One-shot / Few-shot** | Cuando el formato es complejo o poco habitual. |
| **Reglas explícitas** | Cuando hay lógica de negocio que el modelo no puede inferir. |
| **Chain of Thought** | Para tareas de razonamiento, scoring o comparación. |

> **Consejo práctico:** el prompting es iterativo. Prueba, observa dónde falla el modelo y añade la instrucción que le falta. Raramente el primer prompt es el definitivo.

---

## 1.3 ¿Por qué JSON y no texto libre?

### El problema: el texto libre es impredecible

Cuando le pides a un LLM que analice un CV sin especificar formato, puede responderte de formas muy distintas para el mismo candidato, dependiendo del modelo, la versión o incluso el momento:

**Respuesta A:**
```
"Juan García tiene 5 años de experiencia y domina Python y SQL."
```

**Respuesta B:**
```
"El candidato cuenta con aproximadamente 5 años en el sector.
Sus skills incluyen: Python, SQL y Power BI."
```

**Respuesta C:**
```
"Experiencia: 5 años
Nombre: Juan García López
Habilidades técnicas: Python | SQL | Power BI"
```

Las tres contienen la misma información, pero como texto son completamente diferentes. Intentar extraer el nombre o los años con Python requeriría regex frágil que se rompe en cuanto cambia una sola palabra:

```python
# ❌ Parsear texto libre: frágil y poco mantenible
import re

respuesta = "Juan García tiene 5 años de experiencia y domina Python y SQL."

# ¿Y si el modelo dice "cinco años"? ¿O "~5 años"? ¿O "más de 4 años"?
años = re.search(r'(\d+)\s+años', respuesta)
print(años.group(1) if años else "No encontrado")   # → "5" (por suerte)

# Extraer el nombre es aún peor: ¿dónde empieza y dónde acaba?
# No hay forma fiable sin más contexto.
```

Este código funciona para esta frase concreta, pero falla en cuanto el modelo cambia ligeramente su redacción. No escala.

---

### La solución: JSON como contrato fijo

Con JSON defines de antemano qué campos quieres y en qué tipo. El modelo siempre devuelve la misma estructura:

```json
{
    "nombre": "Juan García López",
    "años_experiencia": 5,
    "skills": ["Python", "SQL", "Power BI"]
}
```

Acceder a los datos en Python es directo y no depende de cómo haya redactado la respuesta el modelo:

```python
# ✅ Parsear JSON: robusto y escalable
import json

respuesta_json = '{"nombre": "Juan García López", "años_experiencia": 5, "skills": ["Python", "SQL", "Power BI"]}'
data = json.loads(respuesta_json)   # Convierte el string a diccionario Python

print(data["nombre"])             # → "Juan García López"
print(data["años_experiencia"])   # → 5  (un int, no un string)
print(data["skills"])             # → ["Python", "SQL", "Power BI"]
print(data["skills"][0])          # → "Python"
print(len(data["skills"]))        # → 3
```

Fíjate en algo importante: `años_experiencia` es un **número entero** (`5`), no el string `"5"`. JSON preserva los tipos de dato, lo que significa que puedes hacer operaciones matemáticas directamente, ordenar por score, filtrar por años, etc., sin conversiones.

---

### Comparativa: texto libre vs. JSON

| | Texto libre | JSON |
|---|---|---|
| **Consistencia** | Varía en cada llamada | Misma estructura siempre |
| **Tipos de dato** | Todo llega como string | `int`, `float`, `list`, `bool`... |
| **Acceso a datos** | Regex frágil | `data["campo"]` directo |
| **Listas** | Difíciles de parsear | Arrays nativos: `["a", "b", "c"]` |
| **Valores ausentes** | "N/A", "—", "no consta"... | `null` estándar |
| **Añadir un campo nuevo** | Reescribir el parser | Añadir una clave al JSON |

---

### ¿Cómo forzamos JSON en la API de Gemini?

Usamos dos mecanismos a la vez para máxima fiabilidad:

**1. En el prompt:** pedimos explícitamente JSON y prohibimos cualquier texto adicional
```
"Devuelve ÚNICAMENTE un JSON válido, sin texto adicional, sin bloques de código markdown."
```

**2. En la configuración de la llamada:** fijamos el tipo MIME de la respuesta
```python
config=types.GenerateContentConfig(
    response_mime_type="application/json",   # ← Fuerza JSON puro a nivel de API
    temperature=0.0
)
```

El segundo mecanismo es el decisivo: aunque el modelo "quisiera" añadir un texto introductorio como `"Aquí tienes el resultado:"`, la API lo impide y devuelve solo el JSON. Veremos esto en acción en el Módulo 2.

> **Regla de oro:** el texto libre es para humanos; el JSON es para máquinas. Cuando un LLM extrae datos que va a procesar tu código, siempre pide JSON.

---

## 1.4 Arquitectura del proyecto que vamos a construir

```
┌─────────────────────────────────────────────────────────┐
│                    app.py (Streamlit)                    │
│              Interfaz web en el navegador                │
│         Sube PDFs → Muestra resultados → Descarga Excel  │
└───────────────────┬─────────────────┬───────────────────┘
                    │                 │
         ┌──────────▼──────┐ ┌───────▼──────────┐
         │  pdf_loader.py  │ │   analyzer.py    │
         │  Lee el PDF y   │ │  Llama a Gemini  │
         │  extrae texto   │ │  y devuelve JSON │
         └─────────────────┘ └──────────────────┘
                                       │
                             ┌─────────▼─────────┐
                             │   API de Google   │
                             │   Gemini 2.0 Flash│
                             └───────────────────┘
```

Construiremos de abajo hacia arriba: primero `pdf_loader.py`, luego `analyzer.py`, finalmente `app.py`.

---

## 1.5 Demo en vivo 🎬

> **[INSTRUCTOR]** Antes de que los alumnos empiecen a construir, muestra la aplicación funcionando:  
> 1. Ejecuta `streamlit run app.py` en tu terminal  
> 2. Sube una Job Description de ejemplo  
> 3. Sube 2-3 CVs de prueba  
> 4. Pulsa "Analizar" y deja que el modelo trabaje en tiempo real  
> 5. Muestra la tabla de resultados y descarga el Excel  
> 
> **Objetivo del "wow factor":** que los alumnos vean exactamente qué van a construir en las próximas 2 horas.

---

### 🧪 Mini-ejercicio conceptual (5 min)

Antes de escribir código, reflexiona sobre estas preguntas:

1. ¿Qué ventajas tiene usar un LLM para leer CVs vs. hacerlo con regex o reglas manuales?
2. ¿Qué información de un CV es fácil de extraer automáticamente? ¿Cuál es difícil?
3. ¿Qué riesgos ves en automatizar el filtrado de candidatos con IA?

---

# 🟨 BLOQUE 2 — Construcción Guiada
### ⏱️ 2 horas - 2 horas 30 minutos

---

## ⚙️ Paso 0 — Configuración del entorno

Antes de empezar a programar, necesitamos preparar el entorno de trabajo.

### Estructura de carpetas

Crea esta estructura en tu ordenador:
```
recruitai/
│
├── .env                  ← Tu API Key (NUNCA la compartas)
├── requirements.txt      ← Lista de librerías
├── pdf_loader.py         ← Módulo 1
├── analyzer.py           ← Módulo 2
└── app.py                ← Módulo 3
```

### Obtener la API Key de Google

1. Ve a [https://aistudio.google.com/](https://aistudio.google.com/)
2. Inicia sesión con tu cuenta Google
3. Haz clic en **"Get API Key"** → **"Create API Key"**
4. Copia la clave generada

### Crear el archivo `.env`

En la carpeta `recruitai/`, crea un archivo llamado `.env` (sin extensión) con este contenido:
```
GOOGLE_API_KEY=pega_aquí_tu_clave
```

> ⚠️ **Seguridad:** El archivo `.env` contiene tu clave privada. Nunca lo subas a GitHub. Si usas Git, añade `.env` a tu `.gitignore`.

In [ ]:
# Instalación de dependencias
# Ejecuta esta celda una sola vez

# En Windows (VS Code): py -m pip install -r requirements.txt
# En Mac/Linux:         pip install -r requirements.txt

# O directamente desde aquí:
!pip install google-generativeai pdfplumber pypdf pandas python-dotenv openpyxl xlsxwriter streamlit

In [ ]:
# Verificación: comprueba que la API Key está bien configurada
from dotenv import load_dotenv
import os

load_dotenv()  # Carga las variables del archivo .env

api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print(f"✅ API Key encontrada: {api_key[:8]}...{api_key[-4:]}")
else:
    print("❌ No se encontró GOOGLE_API_KEY. Revisa tu archivo .env")

---

## 📄 Módulo 1 — `pdf_loader.py`
### ⏱️ ~25 minutos

**Responsabilidad única:** recibir un PDF y devolver su texto como string.

### ¿Por qué dos librerías?

No todos los PDFs son iguales. Hay PDFs con texto seleccionable y PDFs que son imágenes escaneadas. Usamos dos librerías como sistema de fallback:

| Librería | Fortaleza | Cuándo usarla |
|----------|-----------|---------------|
| `pdfplumber` | Muy robusta, excelente con PDFs complejos | Primer intento siempre |
| `pypdf` | Más ligera, buen fallback | Cuando pdfplumber falla |

### Funcionamiento de `pdfplumber`

```python
import pdfplumber

with pdfplumber.open("mi_cv.pdf") as pdf:
    for pagina in pdf.pages:          # Itera sobre todas las páginas
        texto = pagina.extract_text() # Extrae el texto de cada página
        print(texto)
```

> **Nota:** `pdfplumber` acepta tanto rutas de archivo (`str`) como objetos de archivo de Python. Esto es importante porque Streamlit nos da objetos de archivo, no rutas.

In [ ]:
# ============================================================
# MÓDULO 1: pdf_loader.py
# ============================================================
# EJERCICIO: Completa la función get_pdf_text()
# 
# La función debe:
# 1. Intentar extraer texto con pdfplumber
# 2. Si falla o el texto está vacío, intentarlo con pypdf
# 3. Devolver None si ambos fallan
# ============================================================

import pdfplumber
import pypdf
import logging

logging.getLogger("pdfminer").setLevel(logging.ERROR)  # Silencia avisos innecesarios

def get_pdf_text(pdf_path):
    """
    Extrae el texto de un PDF.
    
    Args:
        pdf_path: Ruta al PDF (str) o objeto de archivo (de Streamlit)
    
    Returns:
        str: Texto extraído, o None si falla
    """
    text = ""
    
    # --- INTENTO 1: pdfplumber ---
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                try:
                    page_text = page.extract_text()
                    if page_text:                     # Algunas páginas pueden estar vacías
                        text += page_text + "\n"
                except Exception:
                    continue                          # Si una página falla, seguimos con la siguiente
    except Exception:
        text = ""                                    # Resetear para el plan B

    # --- INTENTO 2: pypdf (solo si el intento 1 devolvió vacío) ---
    if not text.strip():
        try:
            reader = pypdf.PdfReader(pdf_path)
            for page in reader.pages:
                try:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + "\n"
                except:
                    continue
        except Exception:
            print("❌ Error crítico: ninguna librería pudo leer el PDF")
            return None

    return text if text.strip() else None


print("✅ Función get_pdf_text() definida correctamente")

In [ ]:
# 🧪 PRUEBA DEL MÓDULO 1
# Sube tu propio CV en la misma carpeta que este notebook y pon aquí el nombre del archivo

NOMBRE_CV = "mi_cv.pdf"  # ← Cambia esto por el nombre de tu archivo

texto = get_pdf_text(NOMBRE_CV)

if texto:
    print(f"✅ PDF leído correctamente ({len(texto)} caracteres)")
    print("\n--- PRIMEROS 500 CARACTERES ---")
    print(texto[:500])
    print("---")
else:
    print("❌ No se pudo leer el PDF. ¿El archivo existe y tiene texto seleccionable?")

### ✅ Checkpoint Módulo 1

Antes de continuar, verifica que:
- [ ] La función `get_pdf_text()` se ejecuta sin errores
- [ ] Al pasarle tu CV, devuelve texto legible (no caracteres extraños)
- [ ] El texto que devuelve se corresponde con el contenido de tu CV

> **⚠️ Problema común:** Si el PDF está escaneado como imagen, `extract_text()` devolverá `None` o texto vacío. En ese caso necesitarías OCR (fuera del alcance de hoy).

---

## 🧠 Módulo 2 — `analyzer.py`
### ⏱️ ~50 minutos

**Responsabilidad:** recibir texto de CV + texto de oferta de trabajo, llamar a la API de Gemini y devolver un diccionario Python con los datos extraídos.

Este es el corazón del proyecto. Vamos a verlo en tres sub-pasos.

### Sub-paso 2.1 — Conectar con la API de Gemini

La librería `google-generativeai` proporciona el cliente:

In [ ]:
# Sub-paso 2.1: Verificar conexión con Gemini
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

# Prueba de conexión con un prompt sencillo
respuesta = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Di exactamente: 'Conexión con Gemini establecida correctamente'"
)

print(respuesta.text)

### Sub-paso 2.2 — Diseñar el prompt de extracción

El prompt es la parte más importante. Debe:
1. Dar un **rol** al modelo ("actúa como...")
2. Definir **exactamente qué extraer**
3. Especificar el **formato de salida** (JSON)
4. Incluir el **contenido** a analizar

Observa cómo el prompt usa **f-strings** de Python para incrustar el texto del CV dinámicamente:

In [1]:
# Sub-paso 2.2: Construir el prompt
# Ejemplo con un CV ficticio para entender la estructura

cv_ejemplo = """
Juan García López
Data Scientist | Madrid

EXPERIENCIA PROFESIONAL:
2022-2024: Data Scientist en Telefónica (2 años)
  - Modelos de churn prediction con Python y scikit-learn
  - Dashboards con Power BI

2019-2022: Analista de Datos en BBVA (3 años)
  - SQL, Tableau, reporting financiero

EDUCACIÓN:
2015-2019: Ingeniería Informática - Universidad Politécnica de Madrid

SKILLS: Python, SQL, scikit-learn, Power BI, Tableau, Git
CERTIFICACIONES: AWS Cloud Practitioner, Google Analytics
"""

oferta_ejemplo = """
Data Scientist - BBVA Innovation Hub
Buscamos Data Scientist con:
- Python avanzado y librerías de ML (scikit-learn, XGBoost)
- SQL y bases de datos relacionales
- Experiencia en sector banca o fintech
- Ubicación: Madrid
"""

# El prompt con f-string para incrustar el contenido
prompt = f"""
Actúa como un Auditor de Talento Senior.
Analiza el CV del candidato y evalúa su afinidad con la oferta de trabajo.

Devuelve ÚNICAMENTE un JSON válido con esta estructura exacta, sin texto adicional:
{{
    "nombre_candidato": "Nombre completo",
    "anos_experiencia_total": 0.0,
    "ubicacion_actual": "Ciudad",
    "Ex_BBVA": "Sí o No",
    "skills_tecnicas": ["skill1", "skill2"],
    "certificaciones": ["cert1"],
    "score_match": 0,
    "justificacion": "Breve explicación del score"
}}

[OFERTA DE TRABAJO]:
{oferta_ejemplo}

[CV DEL CANDIDATO]:
{cv_ejemplo}
"""

print("✅ Prompt construido. Longitud:", len(prompt), "caracteres")
print("\nVista previa del prompt:")
print("-" * 50)
print(prompt[:400] + "...")

✅ Prompt construido. Longitud: 1253 caracteres

Vista previa del prompt:
--------------------------------------------------

Actúa como un Auditor de Talento Senior.
Analiza el CV del candidato y evalúa su afinidad con la oferta de trabajo.

Devuelve ÚNICAMENTE un JSON válido con esta estructura exacta, sin texto adicional:
{
    "nombre_candidato": "Nombre completo",
    "anos_experiencia_total": 0.0,
    "ubicacion_actual": "Ciudad",
    "Ex_BBVA": "Sí o No",
    "skills_tecnicas": ["skill1", "skill2"],
    "certific...


### Sub-paso 2.3 — Llamar a la API y parsear el JSON

Fíjate en el parámetro `response_mime_type="application/json"`. Esto le indica a Gemini que debe devolver **JSON puro**, sin texto introductorio ni bloques de código markdown.

In [ ]:
# Sub-paso 2.3: Llamar a la API y parsear la respuesta
import json
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",  # Fuerza JSON puro
        temperature=0.0                          # 0 = reproducible y consistente
    )
)

# Parsear el JSON de texto a diccionario Python
datos = json.loads(response.text)

print("✅ Respuesta de Gemini parseada correctamente")
print("\nDatos extraídos:")
for campo, valor in datos.items():
    print(f"  {campo}: {valor}")

### 💡 ¿Por qué `temperature=0.0`?

La **temperatura** controla la "creatividad" del modelo:

| Temperatura | Comportamiento | Uso recomendado |
|-------------|----------------|-----------------|
| `0.0` | Respuestas deterministas y consistentes | **Extracción de datos (nuestro caso)** |
| `0.5` | Balance entre consistencia y variedad | Resúmenes, análisis |
| `1.0` | Respuestas más creativas y variadas | Escritura creativa, brainstorming |

In [ ]:
# ============================================================
# MÓDULO 2 COMPLETO: analyzer.py
# ============================================================
# Ahora juntamos todo en una función reutilizable
# ============================================================

import os
import json
from google import genai
from google.genai import types
from dotenv import load_dotenv
from datetime import datetime

def analyze_cv_with_gemini(cv_text, jd_text):
    """
    Analiza un CV contra una oferta de trabajo usando la API de Gemini.
    
    Args:
        cv_text (str): Texto extraído del CV
        jd_text (str): Texto de la oferta de trabajo (Job Description)
    
    Returns:
        dict: Diccionario con los datos extraídos, o None si falla
    """
    load_dotenv()
    api_key = os.getenv("GOOGLE_API_KEY")
    
    if not api_key:
        print("❌ Error: No se encontró GOOGLE_API_KEY en el entorno.")
        return None
    
    client = genai.Client(api_key=api_key)
    year_now = datetime.now().year
    
    prompt = f"""
    Actúa como un Auditor de Talento Senior.
    Analiza el CV del candidato y evalúa su afinidad con la oferta de trabajo.
    El año actual es {year_now}.
    
    REGLAS IMPORTANTES:
    - NO cuentes becas, internships o prácticas como experiencia real
    - Calcula los años desde el PRIMER trabajo real hasta hoy
    - El score debe reflejar el match real con la oferta específica
    
    Devuelve ÚNICAMENTE un JSON válido con esta estructura exacta, sin texto adicional:
    {{
        "nombre_candidato": "Nombre completo",
        "anos_experiencia_total": 0.0,
        "ubicacion_actual": "Ciudad",
        "Ex_BBVA": "Sí o No",
        "categoria_profesional": "Junior / Senior / Manager",
        "skills_tecnicas": ["skill1", "skill2"],
        "certificaciones": ["cert1"],
        "score_match": 0,
        "justificacion": "Breve explicación del score en castellano"
    }}
    
    [OFERTA DE TRABAJO]:
    {jd_text}
    
    [CV DEL CANDIDATO]:
    {cv_text}
    """
    
    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return json.loads(response.text)
    
    except Exception as e:
        print(f"❌ Error en la llamada a Gemini: {e}")
        return None


print("✅ Función analyze_cv_with_gemini() definida correctamente")

In [ ]:
# 🧪 PRUEBA DEL MÓDULO 2 con tu propio CV

NOMBRE_CV = "mi_cv.pdf"  # ← Cambia esto

oferta_prueba = """
Data Scientist - Posición en empresa de consultoría
Buscamos profesional con experiencia en Python, SQL y machine learning.
Se valorará experiencia en sector financiero y banca.
Ubicación: Madrid.
"""

texto_cv = get_pdf_text(NOMBRE_CV)

if texto_cv:
    resultado = analyze_cv_with_gemini(texto_cv, oferta_prueba)
    
    if resultado:
        print("✅ Análisis completado:")
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    else:
        print("❌ El análisis falló")
else:
    print("❌ No se pudo leer el PDF")

### ✅ Checkpoint Módulo 2

Antes de continuar, verifica que:
- [ ] La conexión con la API de Gemini funciona
- [ ] El JSON devuelto tiene todos los campos esperados
- [ ] El nombre extraído corresponde al nombre real en tu CV
- [ ] El score parece razonable para la oferta de prueba

> **💡 Truco de debugging:** Si el JSON está mal formado o le faltan campos, revisa el prompt. Añadir ejemplos concretos de lo que esperas en el JSON suele mejorar mucho los resultados.

---

## 🌐 Módulo 3 — `app.py` (Streamlit)
### ⏱️ ~50 minutos

**Responsabilidad:** interfaz web que une los dos módulos anteriores y presenta los resultados de forma visual.

### ¿Qué es Streamlit?

Streamlit convierte un script Python normal en una aplicación web interactiva, **sin necesidad de HTML, CSS ni JavaScript**.

Se ejecuta con:
```bash
streamlit run app.py
```
Y se abre automáticamente en el navegador en `http://localhost:8501`.

### Componentes básicos de Streamlit que usaremos:

In [ ]:
# Referencia rápida de los componentes de Streamlit que usaremos
# (Este bloque es solo de referencia, no lo ejecutes aquí — ve al archivo app.py)

import streamlit as st

# --- TÍTULOS Y TEXTO ---
st.title("Título principal")
st.subheader("Subtítulo")
st.markdown("Texto con **negrita** o _cursiva_")

# --- LAYOUT EN COLUMNAS ---
col1, col2 = st.columns(2)       # Divide la pantalla en 2 columnas iguales
with col1:
    st.write("Contenido izquierda")
with col2:
    st.write("Contenido derecha")

# --- SUBIR ARCHIVOS ---
archivo = st.file_uploader(
    "Sube un PDF",          # Etiqueta
    type=["pdf"],           # Solo acepta PDFs
    accept_multiple_files=True   # Permite subir varios
)

# --- BOTONES Y MENSAJES ---
if st.button("Analizar"):
    st.success("¡Completado!")  # Verde
    st.warning("Atención")      # Amarillo
    st.error("Error")           # Rojo
    st.info("Información")      # Azul

# --- PROGRESO ---
bar = st.progress(0)            # Barra de progreso (0 a 1)
bar.progress(0.5)               # 50% completado

# --- MOSTRAR DATOS ---
import pandas as pd
df = pd.DataFrame({"A": [1,2], "B": [3,4]})
st.dataframe(df)                # Tabla interactiva

# --- DESCARGAR ARCHIVOS ---
import io
buffer = io.BytesIO()
df.to_excel(buffer, index=False)
st.download_button(
    "📥 Descargar Excel",
    buffer.getvalue(),
    "resultado.xlsx"
)

print("✅ Referencia de componentes Streamlit mostrada")

### Construcción de `app.py`

Ahora vamos a construir el archivo `app.py` completo. Copia este código en un archivo `app.py` en tu carpeta `recruitai/`:

In [ ]:
# ============================================================
# MÓDULO 3 COMPLETO: app.py
# ============================================================
# IMPORTANTE: Este código debe guardarse en app.py y ejecutarse con:
#   streamlit run app.py
#
# NO lo ejecutes directamente en el notebook (Streamlit necesita
# su propio proceso para funcionar correctamente)
# ============================================================

codigo_app = '''
import streamlit as st
import pandas as pd
import io
from datetime import datetime

from pdf_loader import get_pdf_text
from analyzer import analyze_cv_with_gemini

# --- CONFIGURACIÓN DE LA PÁGINA ---
st.set_page_config(
    page_title="RecruitAI",
    page_icon="🤖",
    layout="wide"
)

# --- FUNCIÓN AUXILIAR ---
def parse_list_data(data_field):
    """Convierte listas a string para guardar en Excel."""
    if isinstance(data_field, list):
        return ", ".join(str(item) for item in data_field)
    return str(data_field) if data_field else ""

# --- INTERFAZ ---
st.title("🤖 RecruitAI — Analizador de CVs")
st.markdown("Plataforma de análisis de talento con IA Generativa")
st.markdown("---")

# Dos columnas: JD a la izquierda, CVs a la derecha
col1, col2 = st.columns(2)

with col1:
    st.subheader("1. 📋 Oferta de Trabajo (Job Description)")
    jd_files = st.file_uploader(
        "Sube la oferta en PDF",
        type=["pdf"],
        accept_multiple_files=True,
        key="jd"
    )

with col2:
    st.subheader("2. 👥 CVs de Candidatos")
    cv_files = st.file_uploader(
        "Sube los CVs en PDF",
        type=["pdf"],
        accept_multiple_files=True,
        key="cvs"
    )

# --- BOTÓN DE ANÁLISIS ---
if st.button("🔍 Analizar CVs", type="primary"):
    
    # Validación
    if not jd_files:
        st.error("⚠️ Debes subir al menos una oferta de trabajo.")
        st.stop()
    if not cv_files:
        st.error("⚠️ Debes subir al menos un CV.")
        st.stop()
    
    # Extraer texto de todas las ofertas
    all_jd_text = ""
    for jd in jd_files:
        txt = get_pdf_text(jd)
        if txt:
            all_jd_text += f"\n--- OFERTA: {jd.name} ---\n{txt}\n"
    
    # Procesar cada CV
    results = []
    progress_bar = st.progress(0)
    status_text = st.empty()
    total = len(cv_files)
    hoy = datetime.now().strftime("%d-%m-%Y")
    
    for i, cv_file in enumerate(cv_files):
        progress_bar.progress((i + 1) / total)
        status_text.text(f"Analizando: {cv_file.name} ({i+1}/{total})...")
        
        # Leer texto del CV
        cv_file.seek(0)
        cv_text = get_pdf_text(cv_file)
        
        if not cv_text:
            st.warning(f"⚠️ No se pudo leer: {cv_file.name}")
            continue
        
        # Llamar a la IA
        data = analyze_cv_with_gemini(cv_text, all_jd_text)
        
        if data:
            row = {
                "Fecha": hoy,
                "Archivo": cv_file.name,
                "Nombre": data.get("nombre_candidato", "Desconocido"),
                "Categoría": data.get("categoria_profesional", ""),
                "Ex BBVA": data.get("Ex_BBVA", "No"),
                "Score (0-100)": data.get("score_match", 0),
                "Años Experiencia": data.get("anos_experiencia_total", 0),
                "Ubicación": data.get("ubicacion_actual", ""),
                "Skills Técnicas": parse_list_data(data.get("skills_tecnicas")),
                "Certificaciones": parse_list_data(data.get("certificaciones")),
                "Justificación": data.get("justificacion", "")
            }
            results.append(row)
        else:
            st.error(f"❌ Error analizando: {cv_file.name}")
    
    status_text.text("✅ ¡Análisis completado!")
    
    # --- MOSTRAR RESULTADOS ---
    if results:
        df = pd.DataFrame(results)
        
        # Ordenar por score descendente
        df = df.sort_values(by="Score (0-100)", ascending=False)
        
        st.markdown("---")
        st.subheader(f"📊 Resultados: {len(df)} candidato(s) analizados")
        st.dataframe(df, use_container_width=True)
        
        # Botón de descarga
        buffer = io.BytesIO()
        with pd.ExcelWriter(buffer, engine="xlsxwriter") as writer:
            df.to_excel(writer, index=False, sheet_name="Ranking_CVs")
        
        st.download_button(
            label="📥 Descargar Excel",
            data=buffer.getvalue(),
            file_name=f"ranking_candidatos_{hoy}.xlsx",
            mime="application/vnd.ms-excel"
        )
    else:
        st.error("No se obtuvieron resultados válidos.")
'''

# Guardar el código en app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(codigo_app.strip())

print("✅ app.py creado correctamente")
print("\nPara lanzar la aplicación, ejecuta en tu terminal:")
print("  streamlit run app.py")

### ✅ Checkpoint Módulo 3

Antes de continuar al Bloque 3, verifica que:
- [ ] El archivo `app.py` existe en tu carpeta
- [ ] Al ejecutar `streamlit run app.py` se abre el navegador
- [ ] Los uploaders de PDF aparecen en pantalla
- [ ] Puedes subir un PDF de oferta y un CV de prueba
- [ ] El botón "Analizar" procesa los archivos y muestra resultados
- [ ] El botón de descarga genera un Excel válido

> **⚠️ Problema común:** Si Streamlit no encuentra `pdf_loader.py` o `analyzer.py`, asegúrate de que los tres archivos están en la **misma carpeta** y que ejecutas `streamlit run app.py` desde esa carpeta.

---

# 🟥 BLOQUE 3 — Demo en Vivo y Debate
### ⏱️ 30 minutos

---

## 3.1 Demo: Cada alumno analiza su propio CV

> **[INSTRUCTOR]** Esta parte es experiencial. Cada alumno:
> 1. Sube su propio CV a su instancia de la app
> 2. Usa la oferta de trabajo de prueba que has preparado
> 3. Ve el resultado en tiempo real
> 4. Comparte en voz alta: *"El modelo me dio un score de X y dice que..."*

### Preguntas para el debate en grupo

Una vez que todos han visto su resultado, abrid el debate:

---

## 3.2 Preguntas de Reflexión

### 🎯 Sobre precisión técnica

**¿Los datos extraídos son correctos?**
- ¿El nombre, los años de experiencia y las skills son exactos?
- ¿Qué tipo de errores has observado? (¿sobreestima la experiencia? ¿olvida skills?)
- ¿Qué factores hacen que la extracción sea más o menos fiable?

**¿El score tiene sentido?**
- ¿Estás de acuerdo con el score que te ha dado el modelo para la oferta?
- ¿Cómo cambiaría el score si la oferta fuera diferente?

### ⚖️ Sobre sesgos y ética

**¿Qué sesgos puede tener un sistema así?**
- Los LLMs están entrenados con datos históricos. ¿Qué sesgos de contratación históricos podrían haber absorbido?
- ¿Un CV con muchos anglicismos técnicos puede tener ventaja sobre uno redactado en español?
- ¿Cómo afecta el formato y diseño del CV a la calidad de la extracción de texto?
- ¿Qué pasa con candidatos que tienen gaps en su carrera o trayectorias no lineales?

**¿Es ético usar IA para filtrar candidatos?**
- ¿Quién es responsable si el modelo descarta a un candidato cualificado?
- ¿Debería el candidato saber que su CV lo está leyendo una IA?
- ¿Qué regulación existe en Europa sobre el uso de IA en RRHH? (pista: AI Act)

### 🚀 Sobre mejoras y extensiones

**¿Cómo mejorarías el proyecto?**
- ¿Qué campos adicionales extraerías del CV?
- ¿Cómo harías el sistema más robusto ante CVs con formatos raros?
- ¿Cómo añadirías soporte para CVs en inglés o en otros idiomas?
- ¿Cómo integrarías esto con una base de datos real?

---

## 3.3 Posibles extensiones del proyecto (para casa)

Si quieres llevar el proyecto más allá, aquí tienes ideas ordenadas por dificultad:

| Nivel | Extensión | Pista |
|-------|-----------|-------|
| 🟢 Fácil | Añadir más campos al JSON (idiomas, nivel de inglés, tipo de contrato deseado) | Modifica el prompt en `analyzer.py` |
| 🟢 Fácil | Guardar los resultados en una base de datos SQLite en lugar de Excel | Librería `sqlite3` (incluida en Python) |
| 🟡 Medio | Comparar el mismo CV contra múltiples ofertas y mostrar la mejor | Itera el prompt sobre varias JDs |
| 🟡 Medio | Añadir un gráfico de radar con los scores por categoría | Librería `plotly` con `st.plotly_chart()` |
| 🔴 Difícil | Migrar de Gemini a Claude (API de Anthropic) | Cambia `google.genai` por `anthropic` SDK |
| 🔴 Difícil | Añadir OCR para CVs escaneados | Librería `pytesseract` + Tesseract |

---

## 3.4 Resumen de lo aprendido hoy

En estas 4 horas has construido un sistema completo que:

1. **Lee PDFs automáticamente** con `pdfplumber` y `pypdf`, con sistema de fallback
2. **Se conecta a un LLM** via API (Google Gemini) usando autenticación segura con `.env`
3. **Diseña prompts efectivos** para extracción estructurada de información
4. **Parsea JSON** como protocolo de comunicación entre la IA y tu código Python
5. **Construye una interfaz web** con Streamlit sin necesidad de HTML/CSS
6. **Exporta datos a Excel** para que el resultado sea usable en entornos de negocio

Estos son exactamente los bloques fundamentales de cualquier **aplicación de IA Generativa en producción**.

---

## 📚 Recursos para seguir aprendiendo

| Recurso | URL |
|---------|-----|
| Documentación de Google Gemini | https://ai.google.dev/docs |
| Google AI Studio (obtener API Key) | https://aistudio.google.com/ |
| Documentación de Streamlit | https://docs.streamlit.io/ |
| Prompt Engineering Guide | https://www.promptingguide.ai/es |
| API de Anthropic (Claude) | https://docs.anthropic.com/ |
| AI Act (regulación IA en Europa) | https://artificialintelligenceact.eu/ |